In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from requests.exceptions import HTTPError

BASE_URL = "https://api.openaq.org/v3"

API_KEY = "TU_API_KEY_AQUI"

def get_headers():
    if API_KEY:
        return {"X-API-Key": API_KEY}
    else:
        return {}

DEFINIMOS LOS PARAMETROS DE INTERÉS PM10 Y PM2.5

In [2]:
# pm10 -> id = 1
# pm25 -> id = 2
PARAMS_INTERES = {"pm10": 1, "pm25": 2}
pm_ids = set(PARAMS_INTERES.values())
pm_ids


{1, 2}

DEFINIMOS UNA FUNCIÓN PARA OBTENER TODAS LAS ESTACIONES EN CHILE

In [ ]:
def get_locations_chile_pm(parameters_ids):
    params = {
        "iso": "CL",
        "parameters_id": ",".join(str(p) for p in parameters_ids),
        "limit": 100,
        "page": 1,
        "order_by": "id",
        "sort_order": "asc"
    }

    all_results = []
    while True:
        r = requests.get(f"{BASE_URL}/locations", headers=get_headers(), params=params)
        r.raise_for_status()
        data = r.json()
        all_results.extend(data["results"])

        meta = data["meta"]

        # Page y limit a enteros normales
        try:
            page = int(meta.get("page", 1))
        except:
            page = 1

        try:
            limit = int(meta.get("limit", 100))
        except:
            limit = 100

        # 'found' puede venir como "12345", ">100", etc.
        found_raw = str(meta.get("found", "0"))

        #extraemos los dígitos
        digits = "".join([ch for ch in found_raw if ch.isdigit()])

        if digits.isdigit():
            found = int(digits)
        else:
            # Si no hay dígitos, asumimos que hay más páginas por si acaso
            found = float("inf")

        # ---------------------------------------

        # Condición de término
        if page * limit >= found:
            break

        params["page"] += 1

    return all_results

locations_chile = get_locations_chile_pm(list(PARAMS_INTERES.values()))
len(locations_chile)

100

Función para crear dataframe de sensores obtenidos

In [6]:
def build_sensors_df(locations):
    rows = []
    for loc in locations:
        loc_id = loc["id"]
        loc_name = loc.get("name")
        country_code = loc["country"]["code"]
        locality = loc.get("locality")

        # bounds = [minLon, minLat, maxLon, maxLat]
        bounds = loc.get("bounds")
        if bounds:
            lon_ref = (bounds[0] + bounds[2]) / 2
            lat_ref = (bounds[1] + bounds[3]) / 2
        else:
            lon_ref = None
            lat_ref = None

        for sensor in loc["sensors"]:
            rows.append({
                "location_id": loc_id,
                "location_name": loc_name,
                "locality": locality,
                "country": country_code,
                "sensor_id": sensor["id"],
                "sensor_name": sensor["name"],
                "parameter_id": sensor["parameter"]["id"],
                "parameter_name": sensor["parameter"]["name"],
                "parameter_units": sensor["parameter"]["units"],
                "lat": lat_ref,
                "lon": lon_ref
            })
    return pd.DataFrame(rows)

sensors_df_chile = build_sensors_df(locations_chile)
sensors_pm_chile = sensors_df_chile[sensors_df_chile["parameter_id"].isin(pm_ids)].copy()
len(sensors_pm_chile), sensors_pm_chile.head()

(176,
     location_id     location_name    locality country  sensor_id sensor_name  \
 3            25  Parque O'Higgins    Santiago      CL       1047  pm10 µg/m³   
 4            25  Parque O'Higgins    Santiago      CL       1044  pm25 µg/m³   
 6            26           Inpesca  Talcahuano      CL         45  pm10 µg/m³   
 7            26           Inpesca  Talcahuano      CL      35374  pm25 µg/m³   
 12           27            Concón      Concón      CL      28735  pm10 µg/m³   
 
     parameter_id parameter_name parameter_units        lat        lon  
 3              1           pm10           µg/m³ -33.464142 -70.660797  
 4              2           pm25           µg/m³ -33.464142 -70.660797  
 6              1           pm10           µg/m³ -36.737202 -73.104427  
 7              2           pm25           µg/m³ -36.737202 -73.104427  
 12             1           pm10           µg/m³ -32.924568 -71.515399  )

Función para descargar promedios diarios de un sensor

In [ ]:
def fetch_daily_for_sensor(sensor_id, date_from, date_to, limit=1000,
                           max_retries=6, base_sleep=3):
    """
    Descarga promedios diarios para un sensor dado entre date_from y date_to.
    Maneja rate limit (HTTP 429) con reintentos y backoff.

    date_from/date_to en formato 'YYYY-MM-DD'.
    """
    url = f"{BASE_URL}/sensors/{sensor_id}/days"
    params = {
        "date_from": f"{date_from}T00:00:00Z",
        "date_to":   f"{date_to}T23:59:59Z",
        "limit": limit,
        "page": 1
    }

    results = []

    while True:
        # reintentos por página
        attempt = 0
        while True:
            r = requests.get(url, headers=get_headers(), params=params)
            status = r.status_code

            # caso límite, demasiadas solicitudes
            if status == 429:
                if attempt >= max_retries:
                    print(f"[AVISO] No se pudo obtener datos para sensor {sensor_id} "
                          f"entre {date_from} y {date_to}, página {params['page']} "
                          f"después de {max_retries} reintentos por 429. Se omite esta página.")
                    # devolvemos lo que tengamos hasta ahora
                    return pd.DataFrame(results)

                attempt += 1
                sleep_time = base_sleep * attempt
                print(f"429 en sensor {sensor_id}, año {date_from[:4]} página {params['page']}, "
                      f"reintento {attempt}/{max_retries}, esperando {sleep_time} segundos...")
                time.sleep(sleep_time)
                continue  # vuelve a intentar esta misma página

            # otros errores HTTP → que los maneje afuera
            r.raise_for_status()
            break  # salimos del while interno, tenemos respuesta válida

        data = r.json()

        for item in data["results"]:
            period = item["period"]
            dt_from = period["datetimeFrom"]
            dt_to   = period["datetimeTo"]

            results.append({
                "sensor_id": sensor_id,
                "datetime_from_utc": dt_from["utc"],
                "datetime_from_local": dt_from["local"],
                "datetime_to_utc": dt_to["utc"],
                "datetime_to_local": dt_to["local"],
                "value": item["value"],
                "summary_min":  item.get("summary", {}).get("min"),
                "summary_max":  item.get("summary", {}).get("max"),
                "summary_avg":  item.get("summary", {}).get("avg"),
            })

        meta = data["meta"]
        try:
            page = int(meta.get("page", 1))
        except:
            page = params["page"]

        try:
            limit_page = int(meta.get("limit", limit))
        except:
            limit_page = limit

        found_raw = str(meta.get("found", "0"))
        digits = "".join(ch for ch in found_raw if ch.isdigit())
        if digits.isdigit():
            found = int(digits)
        else:
            #asumimos que si la página viene "incompleta", es la última
            found = page * limit_page

        if page * limit_page >= found:
            break

        params["page"] += 1

    return pd.DataFrame(results)

Función para descargar varios años de un sensor específico

In [11]:
def fetch_sensor_by_years(sensor_id, years):
    all_results = []
    for year in years:
        date_from = f"{year}-01-01"
        date_to   = f"{year}-12-31"
        df_year = fetch_daily_for_sensor(sensor_id, date_from, date_to)
        if not df_year.empty:
            df_year["year"] = year
            all_results.append(df_year)

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        return pd.DataFrame()

DESCARGA MASIVA DE TODAS LAS ESTACIONES DE CHILE QUE MIDEN PM2.5 Y PM10 EN LOS AÑOS 2021-2025

In [12]:
YEARS = [2021, 2022, 2023, 2024, 2025]

all_chile = []

for _, row in tqdm(sensors_pm_chile.iterrows(), total=len(sensors_pm_chile)):
    sid = row["sensor_id"]

    try:
        df_sensor = fetch_sensor_by_years(sid, YEARS)
    except HTTPError as e:
        print(f"Error HTTP en sensor {sid}: {e}")
        continue

    if not df_sensor.empty:
        df_sensor["location_id"] = row["location_id"]
        df_sensor["location_name"] = row["location_name"]
        df_sensor["locality"] = row["locality"]
        df_sensor["country"] = row["country"]
        df_sensor["lat"] = row.get("lat")
        df_sensor["lon"] = row.get("lon")
        df_sensor["parameter_name"] = row["parameter_name"]
        df_sensor["units"] = row["parameter_units"]
        all_chile.append(df_sensor)

    # pequeña pausa para no estresar la API
    time.sleep(0.3)

if all_chile:
    air_quality_chile_2021_2025 = pd.concat(all_chile, ignore_index=True)
else:
    air_quality_chile_2021_2025 = pd.DataFrame()

air_quality_chile_2021_2025.head(), len(air_quality_chile_2021_2025)

  7%|▋         | 12/176 [00:28<05:50,  2.14s/it]

429 en sensor 3013, año 2021 página 1, reintento 1/6, esperando 3 segundos...


 14%|█▎        | 24/176 [00:56<05:32,  2.19s/it]

429 en sensor 1738, año 2021 página 1, reintento 1/6, esperando 3 segundos...
429 en sensor 1738, año 2021 página 1, reintento 2/6, esperando 6 segundos...
429 en sensor 1738, año 2021 página 1, reintento 3/6, esperando 9 segundos...
429 en sensor 1738, año 2021 página 1, reintento 4/6, esperando 12 segundos...
429 en sensor 1738, año 2021 página 1, reintento 5/6, esperando 15 segundos...


 20%|██        | 36/176 [02:07<06:07,  2.62s/it]

...

429 en sensor 8655, año 2021 página 1, reintento 1/6, esperando 3 segundos...
429 en sensor 8655, año 2021 página 1, reintento 2/6, esperando 6 segundos...
429 en sensor 8655, año 2021 página 1, reintento 3/6, esperando 9 segundos...
429 en sensor 8655, año 2021 página 1, reintento 4/6, esperando 12 segundos...
429 en sensor 8655, año 2021 página 1, reintento 5/6, esperando 15 segundos...


 95%|█████████▌| 168/176 [13:07<00:18,  2.37s/it]

429 en sensor 5079416, año 2021 página 1, reintento 1/6, esperando 3 segundos...
429 en sensor 5079416, año 2021 página 1, reintento 2/6, esperando 6 segundos...
429 en sensor 5079416, año 2021 página 1, reintento 3/6, esperando 9 segundos...
429 en sensor 5079416, año 2021 página 1, reintento 4/6, esperando 12 segundos...


100%|██████████| 176/176 [13:58<00:00,  4.77s/it]


(   sensor_id     datetime_from_utc        datetime_from_local  \
 0       1047  2021-01-01T03:00:00Z  2021-01-01T00:00:00-03:00   
 1       1047  2021-01-02T03:00:00Z  2021-01-02T00:00:00-03:00   
 2       1047  2021-01-03T03:00:00Z  2021-01-03T00:00:00-03:00   
 3       1047  2021-01-04T03:00:00Z  2021-01-04T00:00:00-03:00   
 4       1047  2021-01-05T03:00:00Z  2021-01-05T00:00:00-03:00   
 
         datetime_to_utc          datetime_to_local  value  summary_min  \
 0  2021-01-02T03:00:00Z  2021-01-02T00:00:00-03:00   58.7         36.0   
 1  2021-01-03T03:00:00Z  2021-01-03T00:00:00-03:00   33.7         33.0   
 2  2021-01-04T03:00:00Z  2021-01-04T00:00:00-03:00   40.3         25.0   
 3  2021-01-05T03:00:00Z  2021-01-05T00:00:00-03:00   54.4         23.0   
 4  2021-01-06T03:00:00Z  2021-01-06T00:00:00-03:00   56.5         34.0   
 
    summary_max  summary_avg  year  location_id     location_name  locality  \
 0         94.0    58.714287  2021           25  Parque O'Higgins  Sant

In [15]:
len(air_quality_chile_2021_2025)

186835

Guardado csv con data completa

In [13]:
air_quality_chile_2021_2025.to_csv(
    "air_quality_pm25_pm10_Chile_2021_2025.csv",
    index=False
)

Calculo de estaciones can data perdida

In [16]:
total_sensores = sensors_pm_chile["sensor_id"].nunique()
sensores_con_datos = air_quality_chile_2021_2025["sensor_id"].nunique()
sensores_perdidos = total_sensores - sensores_con_datos

total_sensores, sensores_con_datos, sensores_perdidos

(176, 133, 43)

In [ ]:
#convertimos aa datetime forzando utc=True
air_quality_chile_2021_2025["datetime_from_local"] = pd.to_datetime(
    air_quality_chile_2021_2025["datetime_from_local"],
    utc=True,
    errors="coerce"
)

#convertimos a hora local Chile
air_quality_chile_2021_2025["datetime_from_local"] = (
    air_quality_chile_2021_2025["datetime_from_local"]
    .dt.tz_convert("America/Santiago")
)

#Quitamos tz para evitar conflictos en el análisis
air_quality_chile_2021_2025["datetime_from_local_naive"] = (
    air_quality_chile_2021_2025["datetime_from_local"]
    .dt.tz_localize(None)
)

# extraemos el año
air_quality_chile_2021_2025["year"] = (
    air_quality_chile_2021_2025["datetime_from_local_naive"].dt.year
)

In [19]:
air_quality_chile_2021_2025[['datetime_from_local', 'datetime_from_local_naive', 'year']].head()

,datetime_from_local,datetime_from_local_naive,year
0,2021-01-01 00:00:00-03:00,2021-01-01,2021
1,2021-01-02 00:00:00-03:00,2021-01-02,2021
2,2021-01-03 00:00:00-03:00,2021-01-03,2021
3,2021-01-04 00:00:00-03:00,2021-01-04,2021
4,2021-01-05 00:00:00-03:00,2021-01-05,2021


In [20]:
estaciones_con_datos_por_anio = (
    air_quality_chile_2021_2025
    .groupby("year")["location_id"]
    .nunique()
    .sort_index()
)

estaciones_con_datos_por_anio

,location_id
year,
2021,67
2022,72
2023,72
2024,72
2025,71


In [21]:
total_estaciones = sensors_pm_chile["location_id"].nunique()
total_estaciones

100

In [22]:
estaciones_perdidas_por_anio = total_estaciones - estaciones_con_datos_por_anio
estaciones_perdidas_por_anio

,location_id
year,
2021,33
2022,28
2023,28
2024,28
2025,29


Estaciones con datos faltantes

In [ ]:
todas_estaciones = sensors_pm_chile[["location_id", "location_name", "locality", "country"]].drop_duplicates()

faltantes_por_anio = {}

for year in sorted(air_quality_chile_2021_2025["year"].dropna().unique()):
    # estaciones que SÍ tienen datos ese año
    est_con_datos_year = air_quality_chile_2021_2025.loc[
        air_quality_chile_2021_2025["year"] == year, "location_id"
    ].unique()

    # estaciones que NO aparecen ese año
    est_faltantes_year = todas_estaciones[
        ~todas_estaciones["location_id"].isin(est_con_datos_year)
    ].copy()

    faltantes_por_anio[year] = est_faltantes_year

# Ejemplo,ver faltantes 2021
faltantes_por_anio[2021].head(), len(faltantes_por_anio[2021])

(    location_id  location_name locality country
 44           67       Ventanas     None      CL
 65           81  Coronel Norte     None      CL
 69          136    Los Vientos     None      CL
 71          139  Alto Hospicio     None      CL
 72          140  Alto Hospicio     None      CL,
 33)